# 🔄 Pipeline ETL — PI5: Análise de Violência Contra a Mulher

**Objetivo:** Orquestrar o fluxo completo de Extração, Transformação e Carga (ETL) dos dados da SSP-SP.

**Dados:** Ocorrências das DDMs de Sorocaba e Votorantim (2022–Atual)

**Destino:** Google BigQuery — `dados_ssp.dados_ddm`

---

| Etapa | Módulo | Função Principal |
|---|---|---|
| 🔽 Extração | `src.extraction` | `extrair_e_consolidar_dados()` |
| 🔄 Transformação | `src.transformation` | `transformar_dados()` |
| 🔍 Debug | `src.transformation` | `encontrar_valores_nao_numericos()` |
| 🚀 Carga | `src.load` | `carregar_dados_bigquery()` |

In [ ]:
!pip install google-cloud-bigquery pandas-gbq pandas requests openpyxl

## 🔐 Célula 1 — Autenticação Google (Service Account)

> **Dica:** Garanta que o arquivo `credenciais.json` está na raiz do projeto.

In [ ]:
import os
from google.cloud import bigquery

# Define o caminho para o arquivo de credenciais
# O arquivo está na raiz do projeto (uma pasta acima da /notebooks)
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "../credenciais.json"

print("✅ Autenticação configurada! O BigQuery usará '../credenciais.json'.")

## 📦 Célula 2 — Imports dos Módulos src/

In [ ]:
import sys
import os

# Garante que a raiz do projeto está no path do Python,
# permitindo que 'from src.xxx import ...' funcione corretamente.
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# --- Importações dos módulos do projeto ---
from src.extraction import extrair_e_consolidar_dados
from src.transformation import transformar_dados, encontrar_valores_nao_numericos
from src.load import carregar_dados_bigquery, schema_definido

print('✅ Módulos importados com sucesso!')

## ⚙️ Célula 3 — Configurações e Constantes

> **Atenção:** Atualize as variáveis abaixo antes de executar o pipeline.

In [ ]:
import datetime

# --------------------------------------------------
# FONTES DE DADOS — SSP-SP
# --------------------------------------------------
ANO_INICIAL = 2022
ANO_ATUAL = datetime.datetime.now().year

# Gera os links dinamicamente do ano inicial até o ano atual
LINKS_DAS_PLANILHAS = [
    f'https://www.ssp.sp.gov.br/assets/estatistica/transparencia/spDados/SPDadosCriminais_{ano}.xlsx'
    for ano in range(ANO_INICIAL, ANO_ATUAL + 1)
]

# Pasta local onde os arquivos Excel serão salvos
PASTA_DOWNLOADS = '../data/raw'

# --------------------------------------------------
# DESTINO — GOOGLE BIGQUERY
# --------------------------------------------------
NOME_DO_PROJETO = 'projetointegrador5'  # ← ID do projeto GCP
ID_DA_TABELA    = 'dados_ssp.dados_ddm'         # ← dataset.tabela

# --------------------------------------------------
# COLUNAS QUE DEVEM SER NUMÉRICAS (para o debug)
# --------------------------------------------------
COLUNAS_NUMERICAS = ['mes_ocorrencia', 'ano_ocorrencia', 'latitude', 'longitude']

print('✅ Configurações carregadas!')
print(f'   Projeto  : {NOME_DO_PROJETO}')
print(f'   Tabela   : {ID_DA_TABELA}')
print(f'   Arquivos : {len(LINKS_DAS_PLANILHAS)} planilhas configuradas ({ANO_INICIAL}-{ANO_ATUAL})')

## 🔽 Célula 4 — Extração

Baixa os arquivos `.xlsx` da SSP-SP e consolida todas as abas em um único DataFrame.

In [ ]:
dados_consolidados = extrair_e_consolidar_dados(
    lista_de_links=LINKS_DAS_PLANILHAS,
    pasta_downloads=PASTA_DOWNLOADS
)

if dados_consolidados is not None:
    print(f"\n📋 Prévia dos dados brutos ({dados_consolidados.shape[0]} linhas x {dados_consolidados.shape[1]} colunas):")
    display(dados_consolidados.head(3))

## 🔄 Célula 5 — Transformação

Filtra pelas DDMs de Sorocaba e Votorantim, normaliza bairros, cria colunas derivadas e garante os tipos corretos.

In [ ]:
dados_finais = transformar_dados(dados_consolidados)

if dados_finais is not None:
    print(f"\n📋 Prévia dos dados transformados ({dados_finais.shape[0]} linhas x {dados_finais.shape[1]} colunas):")
    display(dados_finais.head(3))

## 🔍 Célula 6 — Debug e Inspeção (Raio-X)

Verifica a integridade do DataFrame antes da carga. Identifica tipos incorretos e valores problemáticos.

In [ ]:
if dados_finais is not None and not dados_finais.empty:

    print('--- RAIO-X DO DATAFRAME FINAL ANTES DA CARGA ---')
    dados_finais.info()

    encontrar_valores_nao_numericos(dados_finais, COLUNAS_NUMERICAS)

    print('\n--- ESTATÍSTICAS DESCRITIVAS ---')
    display(dados_finais.describe(include='all'))

    print('\n--- FIM DO RAIO-X ---')

## 🚀 Célula 7 — Carga para o BigQuery

> ⚠️ **Atenção:** Esta célula usa `WRITE_TRUNCATE` — os dados existentes na tabela serão **substituídos**.

In [ ]:
carregar_dados_bigquery(
    df=dados_finais,
    project_id=NOME_DO_PROJETO,
    table_id=ID_DA_TABELA,
    schema=schema_definido
)